In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()

Saving hf_llm_collector_v5_twoPass.py to hf_llm_collector_v5_twoPass.py


{'hf_llm_collector_v5_twoPass.py': b'#!/usr/bin/env python3\n# v4.1: two-pass (gather \xe2\x86\x92 filter) with Pass1 checkpoint + resume and Drive-safe paths.\n\nimport os, re, time, argparse, threading\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Set, Tuple\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\n\nimport pandas as pd\nfrom tqdm import tqdm\nfrom rapidfuzz import fuzz\nfrom huggingface_hub import HfApi\nfrom huggingface_hub.utils import logging as hf_logging\nhf_logging.set_verbosity_error()\n\nimport logging\nlogging.getLogger("huggingface_hub").setLevel(logging.ERROR)\n\n# ---------- keywords ----------\nCORE_TERMS = ["compliance","security compliance","governance","risk","audit","attestation",\n              "control","controls","control mapping","evidence","policy","policy mining",\n              "assurance","trustworthiness","line-of-defense"]\nFRAMEWORKS  = ["CIS Controls V8","CIS Controls","NIS

In [ ]:
!pip install -q --upgrade --no-deps huggingface_hub rapidfuzz tqdm pyyaml
import sys, pandas, huggingface_hub
print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.7 MB/s eta 0:00:00
Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
pandas: 2.2.2
huggingface_hub: 0.34.4


In [ ]:
import os
os.environ["HUGGING_FACE_HUB_TOKEN"] = "hf_xxxxx" # Please add your own Hugging Facetoken here - Wen 8/31/2025
print("Token set:", os.environ["HUGGING_FACE_HUB_TOKEN"][:6] + "..." + os.environ["HUGGING_FACE_HUB_TOKEN"][-4:])


Token set: hf_OtE...dFLz


Pass 1 — gather broadly (quiet & fast)

1. This covers generative + representation pipelines, no library filter (so formats like gguf/onnx/mlx aren’t excluded).

2. It does not read cardData/README.

In [ ]:
!python hf_llm_collector_v5_twoPass.py pass1 \
  --library '' \
  --pipeline-tags text-generation,text2text-generation,conversational,fill-mask,question-answering,document-question-answering,table-question-answering,text-classification,zero-shot-classification,token-classification,summarization,feature-extraction,sentence-similarity,text-retrieval,translation \
  --sleep 0.03 \
  --checkpoint-interval 20000 \
  --resume \
  --seen /content/drive/MyDrive/hf_seen_pass1.txt \
  --out /content/drive/MyDrive/candidates.csv



[auth] token: hf_OtE...dFLz
[auth] as: miles0827
[pass1] resume: no prior state; starting fresh
[gather] pipeline_tag=text-generation
275738it [2:20:58, 32.60it/s]
[gather] pipeline_tag=text2text-generation
0it [00:00, ?it/s]
[gather] pipeline_tag=conversational
0it [00:00, ?it/s]
[gather] pipeline_tag=fill-mask
15340it [07:53, 32.42it/s]
[gather] pipeline_tag=question-answering
12957it [06:38, 32.51it/s]
[gather] pipeline_tag=document-question-answering
233it [00:07, 31.98it/s]
[gather] pipeline_tag=table-question-answering
166it [00:05, 31.99it/s]
[gather] pipeline_tag=text-classification
100732it [51:42, 32.47it/s]
[gather] pipeline_tag=zero-shot-classification
422it [00:13, 32.26it/s]
[gather] pipeline_tag=token-classification
24022it [12:21, 32.39it/s]
[gather] pipeline_tag=summarization
2467it [01:15, 32.50it/s]
[gather] pipeline_tag=feature-extraction
14190it [07:18, 32.34it/s]
[gather] pipeline_tag=sentence-similarity
12184it [06:15, 32.45it/s]
[gather] pipeline_tag=text-retrie

In [ ]:
!python hf_llm_collector_v5_twoPass.py pass2 \
  --candidates /content/drive/MyDrive/candidates.csv \
  --out /content/drive/MyDrive/results.csv \
  --excluded /content/drive/MyDrive/excluded.csv \
  --workers 5 \
  --checkpoint-interval 2000 \
  --resume \
  --readme-sleep 0.02


[auth] token: hf_OtE...dFLz
[auth] as: miles0827
[pass2] resume: skipping 0 already processed ids
100% 465975/465975 [3:09:52<00:00, 40.90it/s]
[pass2] wrote /content/drive/MyDrive/results.csv (1288), /content/drive/MyDrive/excluded.csv (464687)
